In [0]:
#18 — Validation & Reconciliation Suite (final, syntax-safe)
# Uses:
# - /mnt/output/opa_final_all_levels  (wide Org/ICB/Region from Container 12)
# - opa_final_with_added_metrics      (Container 10 output, org-level wide)
# - df_master_hierarchies, df_icb_region (for fan-out checks)

from pyspark.sql import functions as F
from pyspark.sql.functions import col, sum as s, lit

# -------- Load wide combined output (since Container 13 overwrote final_output to long) --------
final_output_wide = spark.read.parquet("/mnt/output/opa_final_all_levels")

# -------- Helpers --------
def show_sample(df, n=10, title=None):
    if title:
        print("\n" + "=" * len(title))
        print(title)
        print("=" * len(title))
    display(df.limit(n))

def count_failures(df, label):
    c = df.count()
    print(f"{label}: {c} failing row(s)")
    return c

# Pick recent months defensively
month_sel = (
    final_output_wide.select("Der_Activity_Month_Date")
    .where(col("Der_Activity_Month_Date").isNotNull())
    .distinct()
    .orderBy(F.desc("Der_Activity_Month_Date"))
    .limit(3)
)
sample_months = [r[0] for r in month_sel.collect()]

# -------- 1) Identity checks on org-level wide (Container 10) --------
org_base = opa_final_with_added_metrics

checks = []

checks.append((
    "All_Total == All_First + All_FU",
    org_base.filter(
        (col("All_Total").cast("long") != (col("All_First").cast("long") + col("All_FU").cast("long")))
        | col("All_Total").isNull() | col("All_First").isNull() | col("All_FU").isNull()
    )
))

checks.append((
    "All_Proc + All_NoProc == All_Total",
    org_base.filter(
        ((col("All_Proc").cast("long") + col("All_NoProc").cast("long")) != col("All_Total").cast("long"))
        | col("All_Proc").isNull() | col("All_NoProc").isNull() | col("All_Total").isNull()
    )
))

checks.append((
    "F2F_Total == F2F_First + F2F_FU",
    org_base.filter(
        (col("F2F_Total").cast("long") != (col("F2F_First").cast("long") + col("F2F_FU").cast("long")))
        | col("F2F_Total").isNull() | col("F2F_First").isNull() | col("F2F_FU").isNull()
    )
))

checks.append((
    "Remote_Total == Remote_First + Remote_FU",
    org_base.filter(
        (col("Remote_Total").cast("long") != (col("Remote_First").cast("long") + col("Remote_FU").cast("long")))
        | col("Remote_Total").isNull() | col("Remote_First").isNull() | col("Remote_FU").isNull()
    )
))

checks.append((
    "All_2WW == All_First_2WW + All_FU_2WW",
    org_base.filter(
        (col("All_2WW").cast("long") != (col("All_First_2WW").cast("long") + col("All_FU_2WW").cast("long")))
        | col("All_2WW").isNull() | col("All_First_2WW").isNull() | col("All_FU_2WW").isNull()
    )
))

checks.append((
    "All_2WW_DNA == All_First_2WW_DNA + All_FU_2WW_DNA",
    org_base.filter(
        (col("All_2WW_DNA").cast("long") != (col("All_First_2WW_DNA").cast("long") + col("All_FU_2WW_DNA").cast("long")))
        | col("All_2WW_DNA").isNull() | col("All_First_2WW_DNA").isNull() | col("All_FU_2WW_DNA").isNull()
    )
))

print("\n=== Identity checks on Org-level (Container 10) ===")
for label, failing in checks:
    cnt = count_failures(failing, label)
    if cnt > 0:
        show_sample(
            failing.select(
                "Der_Activity_Month_Date", "Adj Org Code", "Treatment_Function_Group",
                "All_Total", "All_First", "All_FU", "All_Proc", "All_NoProc",
                "F2F_Total", "F2F_First", "F2F_FU",
                "Remote_Total", "Remote_First", "Remote_FU",
                "All_2WW", "All_First_2WW", "All_FU_2WW",
                "All_2WW_DNA", "All_First_2WW_DNA", "All_FU_2WW_DNA"
            ),
            n=15,
            title=label + " — examples"
        )

# -------- 2) Join fan-out checks --------
hier_dupes = (
    df_master_hierarchies
    .groupBy("Organisation_Code")
    .agg(F.countDistinct("STP_Code").alias("icb_count"))
    .filter(col("icb_count") > 1)
)
print("\n=== Hierarchy fan-out check (Org -> ICB) ===")
count_failures(hier_dupes, "Orgs mapping to >1 ICB")

icb_dupes = (
    df_icb_region
    .groupBy("ICB_Code")
    .agg(F.countDistinct("Region_Code").alias("region_count"))
    .filter(col("region_count") > 1)
)
print("\n=== ICB->Region fan-out check ===")
count_failures(icb_dupes, "ICBs mapping to >1 Region")

# -------- 3) Rollup reconciliation (Org -> ICB -> Region) using wide data --------
print("\n=== Rollup reconciliation (Org -> ICB -> Region) using wide data ===")

In [0]:
#19  — Explicit rollup delta report (all months) + hard assert on latest
from pyspark.sql import functions as F
from pyspark.sql.functions import col

wide = spark.read.parquet("/mnt/output/opa_final_all_levels")
org = opa_final_with_added_metrics

# make sure we actually have dates
months = (wide.select("Der_Activity_Month_Date")
          .where(col("Der_Activity_Month_Date").isNotNull())
          .distinct()
          .orderBy("Der_Activity_Month_Date"))
print(f"Found {months.count()} month(s) in wide output.")

keys = ["All_Total","All_First","All_FU","F2F_Total","Remote_Total","All_DNA","All_First_DNA","All_FU_DNA"]

# --- Org -> ICB deltas per month ---
org_icb = (
    org.groupBy("Der_Activity_Month_Date","ICB","Treatment_Function_Group")
       .agg(*[F.sum(col(c)).alias(c) for c in keys])
)

icb_rows = (
    wide.filter(col("Level")=="ICB")
        .select("Der_Activity_Month_Date","ICB","Treatment_Function_Group",*keys)
)

org_icb_delta = (
    org_icb.alias("a").join(icb_rows.alias("b"),
        on=["Der_Activity_Month_Date","ICB","Treatment_Function_Group"], how="full")
    .select(
        "Der_Activity_Month_Date",
        *[(col(f"a.{c}")-col(f"b.{c}")).alias(f"delta_{c}") for c in keys]
    )
)

org_icb_summary = org_icb_delta.groupBy("Der_Activity_Month_Date").agg(
    *[F.max(F.abs(col(f"delta_{c}"))).alias(f"ORG2ICB_maxabs_{c}") for c in keys]
)

# --- ICB -> Region deltas per month ---
icb_sum = (
    wide.filter(col("Level")=="ICB")
        .groupBy("Der_Activity_Month_Date","Region_Code","Treatment_Function_Group")
        .agg(*[F.sum(col(c)).alias(c) for c in keys])
)

region_rows = (
    wide.filter(col("Level")=="Region")
        .select("Der_Activity_Month_Date","Region_Code","Treatment_Function_Group",*keys)
)

icb_region_delta = (
    icb_sum.alias("a").join(region_rows.alias("b"),
        on=["Der_Activity_Month_Date","Region_Code","Treatment_Function_Group"], how="full")
    .select(
        "Der_Activity_Month_Date",
        *[(col(f"a.{c}")-col(f"b.{c}")).alias(f"delta_{c}") for c in keys]
    )
)

icb_region_summary = icb_region_delta.groupBy("Der_Activity_Month_Date").agg(
    *[F.max(F.abs(col(f"delta_{c}"))).alias(f"ICB2REG_maxabs_{c}") for c in keys]
)

# --- Show compact table of max |delta| per month ---
summary = (
    org_icb_summary.alias("x")
    .join(icb_region_summary.alias("y"), on="Der_Activity_Month_Date", how="outer")
    .orderBy("Der_Activity_Month_Date")
)

print("\n=== Max |delta| by month (should all be 0 if perfectly consistent) ===")
display(summary)

# --- Hard assert on latest month ---
latest = summary.orderBy(F.desc("Der_Activity_Month_Date")).first()
if latest:
    row = latest.asDict()
    bad = {k:v for k,v in row.items() if k!="Der_Activity_Month_Date" and v not in (0,0.0,None)}
    if bad:
        raise AssertionError(f"Non-zero rollup deltas for latest month {row['Der_Activity_Month_Date']}: {bad}")
    else:
        print(f"✅ Hard assertion passed for latest month {row['Der_Activity_Month_Date']}: all max |delta| == 0")
else:
    print("No months found to assert.")

In [0]:
#20 — Post-run QA (robust): ICB/Region completeness for copy/combos & F2F DNA metrics
from pyspark.sql import functions as F

print("=== Container 17: Post-run QA (read-only, robust) ===")

# ---------- 0) Try to read wide output from C12 (authoritative for column-based checks) ----------
df_wide = None
df_long = None

# Load wide (C12)
try:
    df_wide = spark.read.parquet("/mnt/output/opa_final_all_levels")
    print("Loaded wide table from /mnt/output/opa_final_all_levels")
except Exception as e:
    print("Could not load wide table (/mnt/output/opa_final_all_levels):", str(e))

# Load long (C14) for secondary checks
try:
    df_long = spark.read.parquet("/mnt/output/opa_oprt_final")
    print("Loaded long table from /mnt/output/opa_oprt_final")
except Exception as e:
    if 'df_with_id' in locals():
        df_long = df_with_id
        print("Using in-memory df_with_id for long checks")
    else:
        print("No long table available for OPRT-based checks.")

# ---------- 1) If we have the wide table, run column-based checks there ----------
copy_cols  = [
    "All_DNA1","All_DNA2",
    "All_FU1","All_FU2","All_FU3","All_FU4","All_FU5",
    "All_Total1","All_Total2","All_Total3","All_Total4","All_Total5","All_Total6",
    "Remote_Total1","Remote_Total2"
]
combo_cols = [
    "All_First_plus_All_First_DNA",
    "All_FU_plus_All_FU_DNA",
    "All_Total_plus_All_DNA",
    "F2F_Total_plus_F2F_DNA",
    "Remote_Total_plus_Remote_DNA"
]
extra_cols = ["F2F_DNA", "F2F_DNA_Over_F2F_Total"]

def null_summary(df, label, cols):
    total = df.count()
    print(f"\n--- {label}: {total:,} rows ---")
    if total == 0:
        print("No rows to check.")
        return
    cols_present = [c for c in cols if c in df.columns]
    cols_missing = sorted(set(cols) - set(cols_present))
    if cols_missing:
        print("WARNING: Missing expected columns:", cols_missing)
    if not cols_present:
        print("No expected columns present — skipping null summary.")
        return
    exprs = [F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in cols_present]
    res = df.agg(*exprs).collect()[0].asDict()
    any_nulls = [(k, int(v), round(100*float(v)/total,2)) for k,v in res.items() if v and v > 0]
    if not any_nulls:
        print("All checked columns are fully populated (no NULLs). ✅")
    else:
        print("Columns with NULLs (count, % of rows):")
        for k, v, pct in sorted(any_nulls, key=lambda x: (-x[1], x[0])):
            print(f"  {k:30s} {v:8d} ({pct:5.2f}%)")

def spotcheck_copies(df, label, n=100):
    print(f"\n--- Spot-check copy columns @ {label} (random {n} rows) ---")
    mapping = {
        "All_DNA1":"All_DNA","All_DNA2":"All_DNA",
        "All_FU1":"All_FU","All_FU2":"All_FU","All_FU3":"All_FU","All_FU4":"All_FU","All_FU5":"All_FU",
        "All_Total1":"All_Total","All_Total2":"All_Total","All_Total3":"All_Total",
        "All_Total4":"All_Total","All_Total5":"All_Total","All_Total6":"All_Total",
        "Remote_Total1":"Remote_Total","Remote_Total2":"Remote_Total"
    }
    pairs = [(k,v) for k,v in mapping.items() if k in df.columns and v in df.columns]
    if not pairs:
        print("No copy columns present — skipping.")
        return
    sample = df.orderBy(F.rand()).limit(n)
    exprs = [F.sum(F.when(F.col(k) != F.col(v), 1).otherwise(0)).alias(k+"_neq") for k,v in pairs]
    out = sample.agg(*exprs).collect()[0].asDict()
    bad = [(k,v) for k,v in out.items() if v and v > 0]
    if not bad:
        print("All copy columns match their base values in the sample. ✅")
    else:
        print("Mismatches found in sample:")
        for k,v in bad:
            print(f"  {k}: {v}")

def spotcheck_combos(df, label, n=100):
    print(f"\n--- Spot-check combo columns @ {label} (random {n} rows) ---")
    pairs = {
        "All_First_plus_All_First_DNA": ("All_First","All_First_DNA"),
        "All_FU_plus_All_FU_DNA": ("All_FU","All_FU_DNA"),
        "All_Total_plus_All_DNA": ("All_Total","All_DNA"),
        "F2F_Total_plus_F2F_DNA": ("F2F_Total","F2F_DNA"),
        "Remote_Total_plus_Remote_DNA": ("Remote_Total","Remote_DNA")
    }
    usable = [(newc,a,b) for newc,(a,b) in pairs.items() if all(c in df.columns for c in [newc,a,b])]
    if not usable:
        print("No combo columns present — skipping.")
        return
    sample = df.orderBy(F.rand()).limit(n)
    exprs = [F.sum(F.when(F.col(newc) != (F.col(a).cast("long")+F.col(b).cast("long")), 1).otherwise(0)).alias(newc+"_neq")
             for (newc,a,b) in usable]
    out = sample.agg(*exprs).collect()[0].asDict()
    bad = [(k,v) for k,v in out.items() if v and v > 0]
    if not bad:
        print("All combo columns equal the sum of their parts in the sample. ✅")
    else:
        print("Mismatches found in sample:")
        for k,v in bad:
            print(f"  {k}: {v}")

# Replace the old rate_cols(...) in Container 17 with this:
def rate_cols(df):
    # pick only columns that clearly represent rates (contain '_Over_') or are explicitly named rate fields
    explicit = {"F2F_DNA_Over_F2F_Total", "Remote_DNA_Over_Remote_Total"}
    return sorted([c for c in df.columns if ("_Over_" in c) or (c in explicit)])


def rate_bounds(df, label):
    rcols = rate_cols(df)
    if not rcols:
        print(f"\n--- {label}: No rate columns detected — skipping. ---")
        return
    print(f"\n--- {label}: rate bounds (0–100) ---")
    exprs = [F.sum(F.when((F.col(c) < 0) | (F.col(c) > 100), 1).otherwise(0)).alias(c) for c in rcols]
    out = df.agg(*exprs).collect()[0].asDict()
    bad = [(k,v) for k,v in out.items() if v and v > 0]
    if not bad:
        print("All rates within [0,100] (ignoring NULLs). ✅")
    else:
        print("Out-of-range rate values detected:")
        for k,v in bad:
            print(f"  {k}: {v}")

if df_wide is not None:
    df_icb    = df_wide.filter(F.col("Level") == "ICB")
    df_region = df_wide.filter(F.col("Level") == "Region")

    # 1A) Null coverage
    cols_to_check = [c for c in (copy_cols + combo_cols + extra_cols) if c in df_wide.columns]
    null_summary(df_icb, "ICB level (wide)", cols_to_check)
    null_summary(df_region, "Region level (wide)", cols_to_check)

    # 1B) Spotchecks
    spotcheck_copies(df_icb, "ICB (wide)")
    spotcheck_copies(df_region, "Region (wide)")
    spotcheck_combos(df_icb, "ICB (wide)")
    spotcheck_combos(df_region, "Region (wide)")

    # 1C) Rate bounds
    rate_bounds(df_icb, "ICB (wide)")
    rate_bounds(df_region, "Region (wide)")
else:
    print("\nWide table unavailable — skipping column-based checks.")

# ---------- 2) If we have the long OPRT table, check presence of these metrics by name ----------
if df_long is not None:
    print("\n--- OPRT long checks (by metric name) ---")
    target_metric_names = [  # these are expected as OPRT_Metric_Name entries if they exist upstream
        "All_DNA1","All_DNA2",
        "All_FU1","All_FU2","All_FU3","All_FU4","All_FU5",
        "All_Total1","All_Total2","All_Total3","All_Total4","All_Total5","All_Total6",
        "Remote_Total1","Remote_Total2",
        "All_First_plus_All_First_DNA",
        "All_FU_plus_All_FU_DNA",
        "All_Total_plus_All_DNA",
        "F2F_Total_plus_F2F_DNA",
        "Remote_Total_plus_Remote_DNA",
        "F2F_DNA","F2F_DNA_Over_F2F_Total"
    ]
    present = (df_long
        .filter(F.col("Level").isin("ICB","Region"))
        .groupBy("OPRT_Metric_Name")
        .count()
        .filter(F.col("OPRT_Metric_Name").isin(target_metric_names))
        .orderBy("OPRT_Metric_Name"))

    print("Presence of target metric names at ICB/Region in long table:")
    display(present)

    # Null share by Level for those targets
    mv_nulls = (df_long
        .filter(F.col("Level").isin("ICB","Region"))
        .filter(F.col("OPRT_Metric_Name").isin(target_metric_names))
        .groupBy("Level","OPRT_Metric_Name")
        .agg(F.count("*").alias("rows"),
             F.sum(F.when(F.col("Metric_Value").isNull(),1).otherwise(0)).alias("nulls"),
             F.round(F.sum(F.when(F.col("Metric_Value").isNull(),1).otherwise(0))/F.count("*")*100,2).alias("null_pct"))
        .orderBy("Level","OPRT_Metric_Name"))

    print("Null share for those metrics (long):")
    display(mv_nulls)
else:
    print("\nLong table unavailable — skipping OPRT checks.")

print("\n=== QA complete. ===")

In [0]:
#21 sanity check on nulls in aggregation
from pyspark.sql import functions as F
df = spark.read.parquet("/mnt/output/opa_final_all_levels")

for lvl in ["ICB","Region"]:
    d = df.filter(F.col("Level")==lvl)
    n_null = d.filter(F.col("F2F_DNA_Over_F2F_Total").isNull()).count()
    n_zero = d.filter(((F.col("F2F_Total").isNull()) | (F.col("F2F_Total")==0)) &
                      ((F.col("F2F_DNA").isNull())  | (F.col("F2F_DNA")==0))).count()
    print(lvl, "NULL rates:", n_null, "| zero denominators:", n_zero)
